In [ ]:
# HW07 – Clustering experiments
# Используются датасеты:
# - S07-hw-dataset-02.csv
# - S07-hw-dataset-03.csv
# - S07-hw-dataset-04.csv

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA

# Paths

root = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
data_dir = os.path.join(root, "data")
art = os.path.join(root, "artifacts")
fig = os.path.join(art, "figures")
labels_dir = os.path.join(art, "labels")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(fig, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)


# Preprocessing


def preprocess(df):
    X = df.drop(columns=["sample_id"])

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    if cat_cols:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])
        pre = ColumnTransformer([
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols)
        ])
    else:
        pre = ColumnTransformer([
            ("num", num_pipe, num_cols)
        ])

    return pre, X


# One dataset experiment


def run_dataset(name, df):
    pre, X = preprocess(df)
    Z = pre.fit_transform(X)

    #  KMeans 
    ks = range(2, 16)
    sils = {}
    km_labels = {}

    for k in ks:
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = km.fit_predict(Z)
        sils[k] = silhouette_score(Z, labels)
        km_labels[k] = labels

    best_k = max(sils, key=sils.get)
    best_km_labels = km_labels[best_k]

    plt.figure()
    plt.plot(list(sils.keys()), list(sils.values()), marker="o")
    plt.xlabel("k")
    plt.ylabel("silhouette")
    plt.title(f"{name}: KMeans silhouette vs k")
    plt.savefig(os.path.join(fig, f"{name}_kmeans_silhouette.png"), bbox_inches="tight")
    plt.close()

    #  DBSCAN 
    best_db_score = -1
    best_db_labels = None
    best_db_params = None

    for eps in np.linspace(0.3, 2.0, 8):
        for ms in [5, 10]:
            db = DBSCAN(eps=eps, min_samples=ms)
            labels = db.fit_predict(Z)

            mask = labels != -1
            if mask.sum() < 10 or len(set(labels[mask])) < 2:
                continue

            s = silhouette_score(Z[mask], labels[mask])
            if s > best_db_score:
                best_db_score = s
                best_db_labels = labels
                best_db_params = (eps, ms)

    #  Agglomerative 
    agg_scores = {}
    agg_labels = {}

    for linkage in ["ward", "average"]:
        for k in range(2, 16):
            agg = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            labels = agg.fit_predict(Z)
            agg_scores[(linkage, k)] = silhouette_score(Z, labels)
            agg_labels[(linkage, k)] = labels

    best_linkage, best_k2 = max(agg_scores, key=agg_scores.get)
    best_agg_labels = agg_labels[(best_linkage, best_k2)]

    #  Choose best 
    candidates = {
        "KMeans": (sils[best_k], best_km_labels),
        "DBSCAN": (best_db_score, best_db_labels),
        "Agglomerative": (agg_scores[(best_linkage, best_k2)], best_agg_labels),
    }

    best_method = max(candidates, key=lambda x: candidates[x][0])
    best_labels = candidates[best_method][1]

    #  Metrics 
    def compute_metrics(labels):
        if -1 in labels:
            mask = labels != -1
            return {
                "silhouette": float(silhouette_score(Z[mask], labels[mask])),
                "davies_bouldin": float(davies_bouldin_score(Z[mask], labels[mask])),
                "calinski_harabasz": float(calinski_harabasz_score(Z[mask], labels[mask])),
                "noise_frac": float((labels == -1).mean())
            }
        else:
            return {
                "silhouette": float(silhouette_score(Z, labels)),
                "davies_bouldin": float(davies_bouldin_score(Z, labels)),
                "calinski_harabasz": float(calinski_harabasz_score(Z, labels)),
                "noise_frac": 0.0
            }

    metrics = {
        "KMeans": compute_metrics(best_km_labels),
        "DBSCAN": compute_metrics(best_db_labels) if best_db_labels is not None else None,
        "Agglomerative": compute_metrics(best_agg_labels),
    }

    #  PCA plot 
    pca = PCA(n_components=2, random_state=42)
    Z2 = pca.fit_transform(Z)

    plt.figure()
    plt.scatter(Z2[:, 0], Z2[:, 1], c=best_labels, s=12)
    plt.title(f"{name}: Best method = {best_method}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.savefig(os.path.join(fig, f"{name}_best_pca.png"), bbox_inches="tight")
    plt.close()

    #  Save labels 
    pd.DataFrame({
        "sample_id": df["sample_id"],
        "cluster_label": best_labels
    }).to_csv(os.path.join(labels_dir, f"labels_{name}.csv"), index=False)

    best_cfg = {
        "best_method": best_method,
        "KMeans": {"k": best_k},
        "DBSCAN": {"eps": best_db_params[0], "min_samples": best_db_params[1]} if best_db_params else None,
        "Agglomerative": {"k": best_k2, "linkage": best_linkage},
        "criterion": "max silhouette"
    }

    return metrics, best_cfg


# Run all datasets explicitly


metrics_all = {}
best_cfgs = {}

# Dataset S07-hw-dataset-02.csv
df02 = pd.read_csv(os.path.join(data_dir, "S07-hw-dataset-02.csv"))
metrics_all["ds02"], best_cfgs["ds02"] = run_dataset("ds02", df02)

# Dataset S07-hw-dataset-03.csv
df03 = pd.read_csv(os.path.join(data_dir, "S07-hw-dataset-03.csv"))
metrics_all["ds03"], best_cfgs["ds03"] = run_dataset("ds03", df03)

# Dataset S07-hw-dataset-04.csv
df04 = pd.read_csv(os.path.join(data_dir, "S07-hw-dataset-04.csv"))
metrics_all["ds04"], best_cfgs["ds04"] = run_dataset("ds04", df04)

# Stability check (dataset-02)

pre, X = preprocess(df02)
Z = pre.fit_transform(X)

labels_runs = []
for rs in range(5):
    km = KMeans(
        n_clusters=best_cfgs["ds02"]["KMeans"]["k"],
        random_state=rs,
        n_init=20
    )
    labels_runs.append(km.fit_predict(Z))

aris = [
    adjusted_rand_score(labels_runs[i], labels_runs[j])
    for i in range(5) for j in range(i + 1, 5)
]

best_cfgs["ds02"]["kmeans_ari_mean"] = float(np.mean(aris))

# Save artifacts

with open(os.path.join(art, "metrics_summary.json"), "w") as f:
    json.dump(metrics_all, f, indent=2)

with open(os.path.join(art, "best_configs.json"), "w") as f:
    json.dump(best_cfgs, f, indent=2)
